<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/QSAR/02_RecA_PaDEL_Fingerprint_Matrix_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — PaDEL Fingerprint Matrix Construction

> Clean senior-review version. This notebook contains only the final, logically connected pipeline.
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Calculate PaDEL molecular fingerprints from the curated RecA dataset and export one modeling-ready matrix.

In [8]:
!apt-get install -y default-jre
!pip install -q pandas numpy padelpy

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libnss-mdns fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jre default-jre-headless fonts-dejavu-core
  fonts-dejavu-extra gsettings-desktop-schemas libatk-bridge2.0-0
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
0 upgraded, 19 newly in

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
CURATED_FILE = PROJECT_DIR / "outputs" / "recA_chembl" / "curated" / "recA_curated_ec50_binary.csv"
FP_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "fingerprints"
FP_DIR.mkdir(parents=True, exist_ok=True)

SMILES_FILE = FP_DIR / "recA_padel_input.smi"
FINGERPRINT_FILE = FP_DIR / "recA_padel_fingerprints.csv"
MODELING_MATRIX_FILE = FP_DIR / "recA_modeling_matrix.csv"

### Upload Missing Curated File

The previous cell failed because the file `recA_curated_ec50_binary.csv` was not found at the expected path. Please upload this file using the widget below. After uploading, the file will be moved to the correct location.

In [10]:
from google.colab import files
import shutil

# Upload the file
uploaded = files.upload()

# Assuming the user uploads 'recA_curated_ec50_binary.csv'
# This loop handles cases where the user uploads multiple files, though we expect only one.
for filename in uploaded.keys():
    print(f'User uploaded file "{filename}"')
    # Ensure the target directory exists
    CURATED_FILE.parent.mkdir(parents=True, exist_ok=True)
    # Move the uploaded file to the expected location
    shutil.move(filename, CURATED_FILE)
    print(f'Moved "{filename}" to "{CURATED_FILE}"')

# Verify if the file is now at the correct path
if CURATED_FILE.exists():
    print(f'File successfully placed at {CURATED_FILE}')
else:
    print(f'Error: File still not found at {CURATED_FILE}')


Saving recA_curated_ec50_binary.csv to recA_curated_ec50_binary.csv
User uploaded file "recA_curated_ec50_binary.csv"
Moved "recA_curated_ec50_binary.csv" to "/content/outputs/recA_chembl/curated/recA_curated_ec50_binary.csv"
File successfully placed at /content/outputs/recA_chembl/curated/recA_curated_ec50_binary.csv


## 1. Prepare SMILES input

In [11]:
df = pd.read_csv(CURATED_FILE)
required = ["Name", "SMILES", "bioactivity_class"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

smiles_df = df[["SMILES", "Name"]].dropna().drop_duplicates("Name")
smiles_df.to_csv(SMILES_FILE, sep="\t", header=False, index=False)

print(f"Saved PaDEL input: {SMILES_FILE}")
smiles_df.head()

Saved PaDEL input: /content/outputs/recA_chembl/fingerprints/recA_padel_input.smi


,SMILES,Name
0,Oc1ccc(Nc2nc(-c3ccc4ccccc4c3)cs2)cc1,CHEMBL1076559
1,COC(=O)c1sc(-c2cccs2)cc1NC(=O)/C=C/C(=O)O,CHEMBL1077990
2,Oc1ccc(/N=C/c2ccc(O)c(O)c2)cc1,CHEMBL1093246
3,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,CHEMBL1098875
4,Nc1ccc(O)cc1,CHEMBL1142


## 2. Calculate PaDEL fingerprints

In [13]:
from padelpy import padeldescriptor

padeldescriptor(
    mol_dir=str(SMILES_FILE),
    d_file=str(FINGERPRINT_FILE),
    fingerprints=True,
    # descriptors=False, # Removed as it causes TypeError
    detectaromaticity=True,
    standardizenitro=True,
    standardizetautomers=True,
    threads=2,
    removesalt=True,
    log=True,
)

print(f"Saved fingerprints: {FINGERPRINT_FILE}")

Saved fingerprints: /content/outputs/recA_chembl/fingerprints/recA_padel_fingerprints.csv


## 3. Assemble modeling matrix

In [14]:
fp = pd.read_csv(FINGERPRINT_FILE)

# PaDEL commonly exports molecule name as Name
if "Name" not in fp.columns:
    possible_name_cols = [c for c in fp.columns if c.lower() in ["name", "molecule", "compound"]]
    if possible_name_cols:
        fp = fp.rename(columns={possible_name_cols[0]: "Name"})
    else:
        raise ValueError("Cannot identify compound Name column in PaDEL output.")

meta = df[["Name", "SMILES", "EC50_nM", "pEC50", "bioactivity_class"]].copy()
matrix = meta.merge(fp, on="Name", how="inner")

# Clean fingerprint columns
feature_cols = [c for c in matrix.columns if c not in ["Name", "SMILES", "EC50_nM", "pEC50", "bioactivity_class"]]
matrix[feature_cols] = matrix[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

matrix.to_csv(MODELING_MATRIX_FILE, index=False)

print(matrix.shape)
print(f"Saved modeling matrix: {MODELING_MATRIX_FILE}")
matrix.head()

(2344, 886)
Saved modeling matrix: /content/outputs/recA_chembl/fingerprints/recA_modeling_matrix.csv


,Name,SMILES,EC50_nM,pEC50,bioactivity_class,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL1076559,Oc1ccc(Nc2nc(-c3ccc4ccccc4c3)cs2)cc1,7300.0,5.136677,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL1077990,COC(=O)c1sc(-c2cccs2)cc1NC(=O)/C=C/C(=O)O,15200.0,4.818156,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL1093246,Oc1ccc(/N=C/c2ccc(O)c(O)c2)cc1,380000.0,3.420216,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,380000.0,3.420216,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL1142,Nc1ccc(O)cc1,380000.0,3.420216,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Final output
`outputs/recA_chembl/fingerprints/recA_modeling_matrix.csv`